# VotingAgent Example with Transformers

This notebook demonstrates the `VotingAgent` from `aap_core.orchestration` using Transformers chains.

**Topic:** Which Node.js frontend framework would you choose: Angular, React, or Vue?

Multiple agents with different perspectives will independently generate answers, then the `VotingAgent` will select the best one using BLEU/ROUGE scoring.

In [ ]:
from os import environ

environ.setdefault("HF_HOME", "/data/hf_models/")

'/data/hf_models/'

In [2]:
!export HF_TOKEN=$(cat /data/hf_token.txt)

In [ ]:
from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.agent import BaseAgent
from aap_core.orchestration import VotingAgent
from aap_core.types import AgentMessage, BaseLLMChain
from aap_transformers.chain import ChatCausalMultiTurnsChain

/data/agent_design_pattern/src/transformers/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class Agent(BaseAgent):
    chain: BaseLLMChain

    def execute(self, message: AgentMessage, **kwargs) -> AgentMessage:
        self.state = "running"
        message = self.chain.invoke(message, **kwargs)
        message.execution_result = "success"
        message.origin = self.card.name
        self.state = "idle"
        return message

In [5]:
# Load the shared model once to save memory
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "HuggingFaceTB/SmolLM3-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="cuda")
model.eval()
print(f"Loaded model: {model_name}")

Loading weights: 100%|██████████| 326/326 [00:05<00:00, 58.66it/s]


Loaded model: HuggingFaceTB/SmolLM3-3B


In [6]:
system_prompt = """You are an expert software architect with deep knowledge of frontend frameworks.
Provide a detailed, well-reasoned recommendation."""

user_prompt = "{query}"

# Different perspectives for each agent
perspectives = {
    "react_advocate": "You are a strong advocate for React. Highlight React's strengths including its component model, virtual DOM, ecosystem, and industry adoption. Be fair but clearly favor React.",
    "vue_advocate": "You are a strong advocate for Vue.js. Highlight Vue's strengths including its gentle learning curve, reactivity system, and progressive adoption approach. Be fair but clearly favor Vue.",
    "angular_advocate": "You are a strong advocate for Angular. Highlight Angular's strengths including its full-framework approach, TypeScript integration, dependency injection, and enterprise readiness. Be fair but clearly favor Angular.",
}

# Create agent cards and skills
agent_cards = {}
agent_skills = {}
for name in perspectives:
    skill = AgentSkill(
        id=f'{name}-skill',
        name=f'{name} skill',
        description=f'{name} perspective on frontend frameworks',
        tags=[name]
    )
    card = AgentCard(
        name=name,
        description=f'{name} agent',
        skills=[skill],
        capabilities=AgentCapabilities(),
        default_input_modes=['text'],
        default_output_modes=['text'],
        url="localhost",
        version="0.1.0"
    )
    agent_cards[name] = card
    agent_skills[name] = skill

# Voting agent card
voting_skill = AgentSkill(
    id='voting-skill',
    name='voting skill',
    description='voting skill',
    tags=['voting']
)
voting_card = AgentCard(
    name='voting agent',
    description='voting agent',
    skills=[voting_skill],
    capabilities=AgentCapabilities(),
    default_input_modes=['text'],
    default_output_modes=['text'],
    url="localhost",
    version="0.1.0"
)

print("Created agent cards and skills")

Created agent cards and skills


In [7]:
# Create agents with different perspectives using the SHARED model
agents = []
for name, perspective in perspectives.items():
    chain = ChatCausalMultiTurnsChain(
        model=(tokenizer, model),  # Pass shared model tuple
        system_prompt=perspective,
        user_prompt_template=user_prompt,
        device="cuda",
        max_new_tokens=2048
    )
    agent = Agent(chain=chain, card=agent_cards[name])
    agents.append(agent)
    print(f"Created agent: {name}")

print(f"Total agents: {len(agents)}")

Created agent: react_advocate
Created agent: vue_advocate
Created agent: angular_advocate
Total agents: 3


In [8]:
# Create the VotingAgent using majority_vote with BLEU scoring
voting_agent = VotingAgent(
    agents=agents,
    voting_method="majority_vote",
    scorer="bleu",
    card=voting_card
)
print(f"VotingAgent created with {len(agents)} agents, voting method: majority_vote, scorer: bleu")

VotingAgent created with 3 agents, voting method: majority_vote, scorer: bleu


In [9]:
# Define the query
query = "Which frontend framework would you choose for building a modern web application: Angular, React, or Vue? Consider factors like learning curve, ecosystem, performance, TypeScript support, enterprise adoption, and long-term maintainability. Provide a detailed recommendation with pros and cons of each."
message = AgentMessage(query=query)
print(f"Query: {query}")
print("-" * 80)

Query: Which frontend framework would you choose for building a modern web application: Angular, React, or Vue? Consider factors like learning curve, ecosystem, performance, TypeScript support, enterprise adoption, and long-term maintainability. Provide a detailed recommendation with pros and cons of each.
--------------------------------------------------------------------------------


In [10]:
# Step 1: Execute each perspective agent to generate their opinions
print("=== EXECUTING PERSPECTIVE AGENTS ===")
print()
for agent in agents:
    print(f"Executing agent: {agent.card.name}")
    agent_result = agent.execute(message, temperature=0.7, max_new_tokens=2048)
    if agent_result.execution_result == "success" and len(agent_result.responses) > 0:
        # Append the agent's response to message.responses as (agent_name, response_text)
        message.responses.append((agent.card.name, agent_result.responses[-1][1]))
        print(f"  -> Success: {len(agent_result.responses[-1][1])} characters")
    else:
        print(f"  -> Failed: {agent_result.execution_result}")
    print()

print(f"Total responses collected: {len(message.responses)}")
print("-" * 80)

=== EXECUTING PERSPECTIVE AGENTS ===

Executing agent: react_advocate


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


  -> Success: 5033 characters

Executing agent: vue_advocate
  -> Success: 5031 characters

Executing agent: angular_advocate
  -> Success: 5031 characters

Total responses collected: 6
--------------------------------------------------------------------------------


In [11]:
# Step 2: Execute the VotingAgent to vote on the collected responses
print("=== EXECUTING VOTING AGENT ===")
print()
result = voting_agent.execute(message)
print(f"Execution result: {result.execution_result}")
print(f"Number of candidate responses: {len(result.responses)}")
print("-" * 80)

=== EXECUTING VOTING AGENT ===

Execution result: success
Number of candidate responses: 7
--------------------------------------------------------------------------------


In [12]:
# Display all candidate responses
print("=== CANDIDATE RESPONSES ===")
print()
for agent_name, response in result.responses:
    print(f"--- Agent: {agent_name} ---")
    print(response)
    print()
    print("-" * 80)
    print()

=== CANDIDATE RESPONSES ===

--- Agent: react_advocate ---


When evaluating frontend frameworks for building modern web applications—React, Angular, and Vue—I'll recommend **React** as the top choice, supported by a detailed analysis of pros, cons, and comparisons. Here's a structured breakdown:

---

### **1. React: The Clear Choice**
**Why React?**  
- **Ecosystem and Community**: React leads the ecosystem with over 1 million active developers and an extensive library of third-party tools (e.g., Redux, React Router, Material-UI). Its active community ensures rapid support and innovation.  
- **Performance**: React’s virtual DOM implementation minimizes re-renders, making it the industry standard for performance-critical applications.  
- **TypeScript Support**: While not as deeply integrated as in Angular, React has mature TypeScript support through tools like `@react-tools/react-scripts` and the `@types/react` package.  
- **Enterprise Adoption**: React powers major platforms like 

In [13]:
# Display the winning response (selected by VotingAgent)
print("=== WINNING RESPONSE (selected by VotingAgent) ===")
print()
winner_name, winner_response = result.responses[-1]
print(f"Winner: {winner_name}")
print(winner_response)

=== WINNING RESPONSE (selected by VotingAgent) ===

Winner: voting agent


When evaluating frontend frameworks for building modern web applications—React, Angular, and Vue—I'll recommend **React** as the top choice, supported by a detailed analysis of pros, cons, and comparisons. Here's a structured breakdown:

---

### **1. React: The Clear Choice**
**Why React?**  
- **Ecosystem and Community**: React leads the ecosystem with over 1 million active developers and an extensive library of third-party tools (e.g., Redux, React Router, Material-UI). Its active community ensures rapid support and innovation.  
- **Performance**: React’s virtual DOM implementation minimizes re-renders, making it the industry standard for performance-critical applications.  
- **TypeScript Support**: While not as deeply integrated as in Angular, React has mature TypeScript support through tools like `@react-tools/react-scripts` and the `@types/react` package.  
- **Enterprise Adoption**: React powers major p